# tutorial.ipynb - Day 2 Oxford Tutorial LLM Simulation (Oxford + HBS + Hattie)

> This notebook does NOT call a real LLM API. The Socratic loop uses static if/else to simulate tutor follow-up, avoiding dependency and latency.
> Use with practice.md (deliberate practice) / alignment.md (ILO<->TLA<->AT) / schedule.json (spaced repetition).

## Persona Prompt (Tutor Persona)

You are an Oxford tutorial fellow in **CLV yu liu shi yuce (Customer Lifetime Value + churn prediction: BG/NBD, survival, XGBoost, SHAP)**.
You are also trained in HBS case-method devil's advocacy and Hattie's 4-level formative feedback.

**Hard rules**:
1. **Never give direct answers.** Bu zhijie gei daan. Refuse to say "the answer is X", only ask questions.
2. **Use Socratic questioning.** Each turn use Socratic probes (weishenme/fanli/ruo qianti bian/pingshenme/ruhe - why/counterexample/what-if-premise-changes/on-what-basis/how).
3. **Act as HBS devil's advocate.** When student makes vague claims, play HBS adversary: "What is your evidence? Counterexample?"
4. **Reject vague claims.** "strengthen attention"/"improve experience" type empty phrases get rejected.
5. **End each turn with a probing question.** Each turn must end with one probing question.
6. **Domain anchoring**: Must reference this unit's real dataset (NSW RCT) / libraries (pandas/sklearn) / concepts (BG/NBD, RFM, AUC-ROC, class_weight, stratify, discount_factor).


## Pre-Tutorial Task (Forced Retrieval, submit 24h before)

> Before tutorial starts, student must independently submit a 300-word essay answering the following. No submission = tutorial cancelled.

**Essay prompts (choose one, all must be completed over the unit)**:

1. **CLV three methods comparison**: A B2B SaaS customer pays 120k/year, 3-year retention 0.85, discount rate 0.1. Estimate 5-year expected CLV using historical CLV / simple predictive CLV / BG/NBD simplified CLV respectively. Which is least credible in B2B? Why?

2. **Churn evaluation blindspot**: You trained LogisticRegression churn model, accuracy=0.92, but churners are only 8%. Is 0.92 credible? Which two metrics should you switch to? If RandomForest AUC=0.55, explain from NSW data feature dimension.

3. **Q1 retention decision**: Q1 (high CLV x high churn risk) has 500 customers, avg CLV 3000 yuan, per-customer retention cost 200 yuan, success rate 30%, marketing budget 100k yuan. Write the decision inequality `retained_value = N x success x CLV` vs `cost = N x cost_per`. Should you do it?

**Submission**: Write to `student_essay.md`, tutor reads 1h before tutorial.

**Why forced retrieval**: Roediger & Karpicke (2006) empirically show retrieval practice beats rereading by 50%+ on long-term memory. Tutorial is not lecture, it is being probed.


In [ ]:
# Socratic Multi-Turn Loop (static if/else simulation, no real LLM API call)
# Design: >=4 turns, each turn contains >=1 Socratic question
# Socratic question keywords: weishenme(why)/fanli(counterexample)/ruo qianti bian(what if premise changes)/pingshenme(on what basis)/ruhe(how)

import json, os

STUDENT_ESSAY = {
    "method_choice": "B2B uses historical CLV - simplest, most credible",
    "metric_choice": "0.92 accuracy is good, model usable",
    "q1_decision": "Q1 should be retained because CLV is high"
}

# Tutor probe bank (5 Socratic questions with required keywords)
TUTOR_QUESTIONS = {
    "Q1_weishenme": "Weishenme (why) do you think historical CLV is most credible in B2B? Historical CLV only looks at past profit - can it predict future? What happens to BG/NBD's Poisson purchase-rate assumption under B2B contract cycle?",
    "Q2_fanli": "Gei ge fanli (counterexample): A customer spent 120k/year for 3 years, but next month contract expires and competitor bids 20% lower. Historical CLV=360k, but true future CLV could be 0. What signal does historical CLV miss?",
    "Q3_ruoqiantibian": "Ruo qianti bian (if premise changes): churn rate rises from 8% to 40% (SaaS industry crisis), does your accuracy=0.92 still hold? How do Precision/Recall shift under class_weight='balanced'? Pingshenme (on what basis) do you still trust accuracy?",
    "Q4_pingshenme": "Pingshenme (on what basis) do you say 'Q1 should be retained'? Your evidence is 'CLV is high', but 500 x 200 = 100k cost, 30% success means 350 fail. The 350's cost is sunk - pingshenme go? Write retained_value vs cost inequality.",
    "Q5_ruhe": "Ruhe (how) to lift NSW AUC~0.54 to industrial AUC>0.80? What features must be added? Weishenme does randomization make baseline features weak predictors of churn? How does this connect to Day 1 RCT logic?"
}

# Student responses (static simulation; in real use student types each)
STUDENT_RESPONSES = {
    "Q1_weishenme": "Historical CLV is simple, no assumptions needed",
    "Q2_fanli": "Historical CLV misses contract expiry signal",
    "Q3_ruoqiantibian": "Then accuracy might not be credible",
    "Q4_pingshenme": "retained_value=500x0.3x3000=450k > cost=100k, so do it",
    "Q5_ruhe": "Need to add login frequency / session duration / customer service complaints"
}

# Static Socratic loop: 5 turns (>=4 required), each = tutor question + student answer + tutor assessment
def socratic_turn(turn_id, question, student_answer):
    print(f"\n=== Turn {turn_id} ===")
    print(f"[TUTOR]: {question}")
    print(f"[STUDENT]: {student_answer}")
    # Static assessment: check answer depth
    if len(student_answer) < 15 or "simple" in student_answer.lower() or "no assumptions" in student_answer.lower():
        assessment = "[HBS Devil's Advocate] Answer too shallow. What is your evidence? Counterexample?"
    elif "retained_value" in student_answer or "500" in student_answer:
        assessment = "[Oxford] Inequality correct, but what about the 350 failures' sunk cost? How do you handle it?"
    elif "login frequency" in student_answer or "customer service" in student_answer:
        assessment = "[Oxford] Direction right, but pingshenme are these features more predictive than baseline demographics? How does NSW randomization support your claim?"
    else:
        assessment = "[Oxford] Continue. Weishenme? Fanli? Ruo qianti bian?"
    print(f"[TUTOR ASSESSMENT]: {assessment}")
    return assessment

# Execute 5-turn Socratic loop (>=4 turns, 5 Socratic questions)
turn_results = []
for i, (qid, q) in enumerate(TUTOR_QUESTIONS.items(), 1):
    a = STUDENT_RESPONSES.get(qid, "")
    r = socratic_turn(i, q, a)
    turn_results.append({"turn": i, "question_id": qid, "answer": a, "assessment": r})

print(f"\n[Socratic Loop Done] {len(turn_results)} turns, 5 Socratic probes (weishenme/fanli/ruo qianti bian/pingshenme/ruhe)")


In [ ]:
# Student Model Read/Write (records mastery/blind_spots, persists across sessions)
# Fields: mastery (0-1 per ILO), blind_spots (list), last_session, retrieval_strength

import json, os, datetime

STUDENT_MODEL_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "U-E2-Day2",
        "mastery": {"ILO1_CLV": 0.0, "ILO2_RFM": 0.0, "ILO3_churn": 0.0, "ILO4_matrix": 0.0},
        "blind_spots": [],
        "last_session": None,
        "sessions_count": 0,
        "retrieval_strength": {"C1": 0.0, "C2": 0.0, "C3": 0.0, "C4": 0.0, "C5": 0.0}
    }

def update_student_model(model, turn_results):
    # Update mastery based on Socratic loop answer quality
    # Shallow answer (-0.1), contains formula (+0.2), contains specific features (+0.3)
    for tr in turn_results:
        qid = tr["question_id"]
        ans = tr["answer"]
        if "Q1" in qid or "Q2" in qid:
            model["mastery"]["ILO1_CLV"] = min(1.0, model["mastery"]["ILO1_CLV"] + (0.2 if "formula" in ans or "BG/NBD" in ans else -0.1))
        elif "Q3" in qid:
            model["mastery"]["ILO3_churn"] = min(1.0, model["mastery"]["ILO3_churn"] + (0.2 if "accuracy" in ans else -0.1))
        elif "Q4" in qid:
            model["mastery"]["ILO4_matrix"] = min(1.0, model["mastery"]["ILO4_matrix"] + (0.3 if "retained_value" in ans else -0.1))
        elif "Q5" in qid:
            model["mastery"]["ILO3_churn"] = min(1.0, model["mastery"]["ILO3_churn"] + (0.3 if "login frequency" in ans else -0.1))
    # Blind spots: ILO with mastery < 0.5
    model["blind_spots"] = [ilo for ilo, m in model["mastery"].items() if m < 0.5]
    model["last_session"] = datetime.datetime.now().isoformat()
    model["sessions_count"] += 1
    return model

model = load_student_model()
model = update_student_model(model, turn_results)

with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
    json.dump(model, f, ensure_ascii=False, indent=2)

print("[student_model.json updated]")
print(json.dumps(model, ensure_ascii=False, indent=2))
print(f"\n[Blind spot diagnosis] ILO with mastery < 0.5: {model['blind_spots']}")
print(f"[Suggestion] For blind spots, fall back to practice.md corresponding drill Worked example")


## Hattie 4-Level Formative Feedback (based on student_model.json blind spots)

> Hattie & Timperley (2007) 4-level feedback. **Avoid Self-level praise** (Self-level feedback like "you're smart" is ineffective, Hattie d=0.14).

### [TASK] Level Feedback (task-level, answers "did I do it right")
- **ILO1 CLV**: In your BG/NBD simplified formula `pred_clv = F x retention^12 x AOV x 12 x discount_factor`, is `discount_factor` correct? Check `discount_factor = 1/(1+0.1)^t` is per-month discount, not `=1.0` (undiscounted). NSW TODO3 unit test will catch this.
- **ILO2 RFM**: In your `pd.qcut` binning, is Recency label reversed `[4,3,2,1]`? If ascending `[1,2,3,4]` then Champions/Lost labels are flipped. Check TODO2 five-class sample counts - if Lost = full count then label is reversed.
- **ILO3 churn**: AUC=0.54 on NSW is expected (baseline features randomized), but if you didn't set `class_weight='balanced'`, Recall approaches 0 (predict all non-churn). Check classification_report Recall(positive class).

### [PROCESS] Level Feedback (process-level, answers "how should I do it")
- **Data leakage check**: Did your `StandardScaler.fit(X)` fit on full set or `fit(X_train)` then `transform(X_test)`? Former = data leakage, test set info pollutes training. Correct approach in solution.ipynb TODO4.
- **stratify choice**: `train_test_split(X, y, stratify=y)` or `stratify=None`? NSW churn ~20%, without stratify test set distribution drifts, AUC estimate high variance. **Correct** = `stratify=y`.
- **BG/NBD assumption violation handling**: B2B contract cycle violates Poisson purchase-rate assumption. **Process strategies**: (1) switch to Bayesian CLV (PyMC) with contract-cycle prior (2) align contract cycle before BG/NBD input (3) accept violation, explicitly note limitation in 300-word analysis.

### [SELF-REG] Level Feedback (self-regulation level, answers "how do I monitor myself")
- **Blind spot self-check**: Which ILO in your `student_model.json` has mastery < 0.5? If ILO3 < 0.5, can you independently write the three traps (stratify/scaler/class_weight) without looking at `solution.ipynb`? No = trigger practice.md weak_loop, fall back to D3-BGNBD Worked.
- **Retreat strategy**: When stuck on D5-SKLEARN Independent stage over 20 minutes, proactively retreat to Faded stage, don't grind. **Self-regulation** = knowing when to retreat.
- **Metacognitive prompt**: Every 15 minutes ask yourself "Is the step I'm doing now training an ILO or just running code?" Running through != understanding.

### [FEED-FORWARD] Level Feedback (feed-forward level, answers "where next")
- **If ILO1 mastery >= 0.8**: Next go to Day 3 (MMM/MTA/incremental measurement), combine CLV with channel attribution, form "prediction + prescription" loop.
- **If ILO3 mastery < 0.5**: Next revisit practice.md D5-SKLEARN Worked, then re-validate with D1.2 pre-test. Still stuck = tutor intervention to diagnose prerequisite (usually one of OLS / logistic regression / evaluation metrics).
- **If ILO4 mastery >= 0.8**: Next study uplift modeling (Skill 3 causal inference), upgrade from "predict who churns" to "predict causal effect of intervention".
- **Cross-unit review**: schedule.json C1-C5 spaced repetition due dates, next review at +1/+3/+8/+21/+60/+180 days, scheduled by FSRS-6 algorithm.


## Daily Limit and Exit Artifact

### Daily Limit (prevent LLM dependency)
- **At most 1 tutorial session per unit per day** (this notebook = 1 session/day)
- Rationale: Hattie (2009) meta-analysis shows over-reliance on external feedback weakens self-regulation (Self-Reg level feedback fails). Tutorial is scaffolding, not a crutch.
- **Forced cooldown**: After this session ends, running this notebook again within 24h will prompt "Today's tutorial quota used, please do practice.md Independent stage first".
- **Exit condition**: When all 4 ILOs in `student_model.json` reach mastery >= 0.7, this unit's tutorial unlocks to "on-demand" mode (no daily limit, because you can self-regulate).

### Exit Artifact (must produce before tutorial counts as complete)

Write the following 3 items into `tutorial_exit.md` (submit to count as done):

1. **2-3 blind spots** (copy from `student_model.json` blind_spots field, add one sentence "why I got it wrong"):
   - Example: "ILO3_churn mastery=0.3 - I treated accuracy=0.92 as good model, forgot 8% churn rate makes Accuracy unreliable, should switch to AUC-ROC + Precision/Recall"
   - Example: "ILO1_CLV mastery=0.4 - I wrote BG/NBD simplified formula with `discount_factor=1.0`, didn't realize long-term CLV must discount"

2. **Recommended review units** (based on blind spots):
   - ILO1 blind -> review practice.md D3-BGNBD Worked + schedule.json C1/C5 (spaced repetition due +1 day)
   - ILO2 blind -> review practice.md D2-RFM Worked + schedule.json C2
   - ILO3 blind -> review practice.md D5-SKLEARN Worked + schedule.json C3 + notes "why NSW AUC~0.54 is expected"
   - ILO4 blind -> review practice.md D6-MATRIX Worked + schedule.json C4 + D1.3 decision inequality

3. **One Socratic question** (a question you ask back at the tutor, must reference this unit's real concepts):
   - Example: "If NSW data with login frequency added lifts AUC to 0.82, does that prove login frequency causally drives churn? Or just correlation? How does this relate to Day 1 RCT randomization?"
   - Example: "BG/NBD assumes customer independence, but B2B SaaS has org accounts shared by multiple people - does this violate independence? How to handle with Bayesian hierarchical model?"

### Exit Confirmation

```python
# Pre-exit self-check (run after writing tutorial_exit.md)
import json, os
m = json.load(open("student_model.json", encoding="utf-8"))
assert os.path.exists("tutorial_exit.md"), "tutorial_exit.md not submitted"
assert m["sessions_count"] >= 1, "Socratic loop not completed"
print(f"[Tutorial complete] blind_spots={m['blind_spots']}, next review due per schedule.json")
print("[Daily limit] Today's quota used, next session in 24h. Prioritize practice.md Independent stage")
```
